In [55]:
import pandas as pd
import time
from nba_api.stats.endpoints import leaguegamelog, teamestimatedmetrics, leaguedashplayerstats, boxscoretraditionalv2

# ----------------------------------
# Step 1: Pull Game Logs + Team Stats
# ----------------------------------

def get_game_logs_for_season(season: str, season_type: str = "Regular Season"):
    print(f"Fetching game logs for {season}...")
    games = leaguegamelog.LeagueGameLog(season=season, season_type_all_star=season_type)
    df = games.get_data_frames()[0]
    df["Season"] = season
    return df

def get_team_metrics_for_season(season: str):
    print(f"Fetching team metrics for {season}...")
    metrics = teamestimatedmetrics.TeamEstimatedMetrics(season=season)
    df = metrics.get_data_frames()[0]
    df["Season"] = season
    return df

def pull_all_game_and_team_data(start_year=2015, end_year=2025):
    seasons = [f"{year}-{str(year+1)[-2:]}" for year in range(start_year, end_year + 1)]
    all_game_logs = []
    all_team_metrics = []

    for season in seasons:
        try:
            game_log = get_game_logs_for_season(season)
            team_metrics = get_team_metrics_for_season(season)
            all_game_logs.append(game_log)
            all_team_metrics.append(team_metrics)
            print(f"Completed season {season}")
            time.sleep(1.5)  # basic rate limit protection
        except Exception as e:
            print(f"Error pulling season {season}: {e}")
            time.sleep(5)

    games_df = pd.concat([df for df in all_game_logs if not df.empty], ignore_index=True)
    teams_df = pd.concat([df for df in all_team_metrics if not df.empty], ignore_index=True)

    return games_df, teams_df

# ----------------------------------
# Step 2: Create Game-Level Dataset
# ----------------------------------

def create_game_level_data(game_logs):
    games = []
    grouped = game_logs.groupby("GAME_ID")

    for game_id, group in grouped:
        if len(group) != 2:
            continue

        home_row = group[group["MATCHUP"].str.contains("vs.")]
        away_row = group[group["MATCHUP"].str.contains("@")]

        if home_row.empty or away_row.empty:
            continue

        game_data = {
            "GameID": game_id,
            "Season": group["Season"].iloc[0],
            "GameDate": group["GAME_DATE"].iloc[0],
            "HomeTeam": home_row["TEAM_NAME"].values[0],
            "AwayTeam": away_row["TEAM_NAME"].values[0],
            "HomePTS": home_row["PTS"].values[0],
            "AwayPTS": away_row["PTS"].values[0],
            "TotalPoints": home_row["PTS"].values[0] + away_row["PTS"].values[0]
        }

        games.append(game_data)

    return pd.DataFrame(games)

def merge_team_metrics(game_df, team_metrics_df):
    merged = pd.merge(
        game_df,
        team_metrics_df.add_prefix("Home_"),
        left_on=["HomeTeam", "Season"],
        right_on=["Home_TEAM_NAME", "Home_Season"],
        how="left"
    )

    merged = pd.merge(
        merged,
        team_metrics_df.add_prefix("Away_"),
        left_on=["AwayTeam", "Season"],
        right_on=["Away_TEAM_NAME", "Away_Season"],
        how="left"
    )

    return merged

# ----------------------------------
# Step 3: Add Rolling 5, 7, 10-Game Averages
# ----------------------------------

def add_rolling_averages(merged_games):
    merged_games["GameDate"] = pd.to_datetime(merged_games["GameDate"])
    
    # Sort by HomeTeam to calculate Home rolling stats
    merged_games = merged_games.sort_values(["HomeTeam", "GameDate"])
    for window in [5, 7, 10]:
        merged_games[f"Home_Rolling_Pace_{window}g"] = merged_games.groupby("HomeTeam")["Home_E_PACE"].shift(1).rolling(window).mean()
        merged_games[f"Home_Rolling_OffRating_{window}g"] = merged_games.groupby("HomeTeam")["Home_E_OFF_RATING"].shift(1).rolling(window).mean()
        merged_games[f"Home_Rolling_DefRating_{window}g"] = merged_games.groupby("HomeTeam")["Home_E_DEF_RATING"].shift(1).rolling(window).mean()
        merged_games[f"Home_Rolling_NetRating_{window}g"] = merged_games.groupby("HomeTeam")["Home_E_NET_RATING"].shift(1).rolling(window).mean()
        merged_games[f"Home_Rolling_TOVPct_{window}g"] = merged_games.groupby("HomeTeam")["Home_E_TM_TOV_PCT"].shift(1).rolling(window).mean()
        merged_games[f"Home_Rolling_REBPct_{window}g"] = merged_games.groupby("HomeTeam")["Home_E_REB_PCT"].shift(1).rolling(window).mean()

    # Sort by AwayTeam to calculate Away rolling stats
    merged_games = merged_games.sort_values(["AwayTeam", "GameDate"])
    for window in [5, 7, 10]:
        merged_games[f"Away_Rolling_Pace_{window}g"] = merged_games.groupby("AwayTeam")["Away_E_PACE"].shift(1).rolling(window).mean()
        merged_games[f"Away_Rolling_OffRating_{window}g"] = merged_games.groupby("AwayTeam")["Away_E_OFF_RATING"].shift(1).rolling(window).mean()
        merged_games[f"Away_Rolling_DefRating_{window}g"] = merged_games.groupby("AwayTeam")["Away_E_DEF_RATING"].shift(1).rolling(window).mean()
        merged_games[f"Away_Rolling_NetRating_{window}g"] = merged_games.groupby("AwayTeam")["Away_E_NET_RATING"].shift(1).rolling(window).mean()
        merged_games[f"Away_Rolling_TOVPct_{window}g"] = merged_games.groupby("AwayTeam")["Away_E_TM_TOV_PCT"].shift(1).rolling(window).mean()
        merged_games[f"Away_Rolling_REBPct_{window}g"] = merged_games.groupby("AwayTeam")["Away_E_REB_PCT"].shift(1).rolling(window).mean()

    return merged_games

# ----------------------------------
# Step 4: Add Rest Days / Back-to-Back Flags
# ----------------------------------

def compute_rest_days(game_logs):
    game_logs["GAME_DATE"] = pd.to_datetime(game_logs["GAME_DATE"])
    game_logs = game_logs.sort_values(["TEAM_NAME", "GAME_DATE"])
    game_logs["Prev_Game_Date"] = game_logs.groupby("TEAM_NAME")["GAME_DATE"].shift(1)
    game_logs["Rest_Days"] = (game_logs["GAME_DATE"] - game_logs["Prev_Game_Date"]).dt.days
    game_logs["BackToBack"] = (game_logs["Rest_Days"] == 1).astype(int)
    return game_logs

def merge_rest_b2b(merged_games, game_logs):
    home_rest = game_logs.add_prefix("Home_")
    merged = pd.merge(
        merged_games,
        home_rest[["Home_TEAM_NAME", "Home_GAME_DATE", "Home_Rest_Days", "Home_BackToBack"]],
        left_on=["HomeTeam", "GameDate"],
        right_on=["Home_TEAM_NAME", "Home_GAME_DATE"],
        how="left"
    )

    away_rest = game_logs.add_prefix("Away_")
    merged = pd.merge(
        merged,
        away_rest[["Away_TEAM_NAME", "Away_GAME_DATE", "Away_Rest_Days", "Away_BackToBack"]],
        left_on=["AwayTeam", "GameDate"],
        right_on=["Away_TEAM_NAME", "Away_GAME_DATE"],
        how="left"
    )

    return merged

# ----------------------------------
# Step 5: Starter Detection + Missing Players
# ----------------------------------

def get_true_starters(season="2024-25", top_n=5):
    print(f"Fetching player stats for {season} for starter detection...")
    players = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        per_mode_detailed="PerGame"
    )
    player_df = players.get_data_frames()[0]

    player_df["Starter_Score"] = (
        player_df["MIN"] * 0.65 +
        player_df["PTS"] * 0.25 +
        player_df["PLUS_MINUS"] * 0.10
    )

    top_starters = {}
    for team, group in player_df.groupby("TEAM_ABBREVIATION"):
        group_sorted = group.sort_values(by="Starter_Score", ascending=False)
        top_names = group_sorted["PLAYER_NAME"].head(top_n).tolist()
        top_starters[team] = top_names

    return top_starters

def detect_missing_starters(games_df, top_starters):
    def check_starters(game_id, home_team, away_team):
        try:
            box = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id)
            player_stats = box.get_data_frames()[0]

            missing_home = 0
            missing_away = 0

            if home_team in top_starters:
                for player in top_starters[home_team]:
                    player_row = player_stats[player_stats["PLAYER_NAME"] == player]
                    if player_row.empty or player_row["MIN"].iloc[0] in ["0", "0:00", 0]:
                        missing_home += 1

            if away_team in top_starters:
                for player in top_starters[away_team]:
                    player_row = player_stats[player_stats["PLAYER_NAME"] == player]
                    if player_row.empty or player_row["MIN"].iloc[0] in ["0", "0:00", 0]:
                        missing_away += 1

            return pd.Series([missing_home, missing_away])

        except Exception as e:
            print(f"Error fetching box score for {game_id}: {e}")
            return pd.Series([None, None])

    games_df[["KeyPlayersOut_Home", "KeyPlayersOut_Away"]] = games_df.apply(
        lambda row: check_starters(row["GameID"], row["HomeTeam"], row["AwayTeam"]),
        axis=1
    )

    return games_df

# ----------------------------------
# Main Execution
# ----------------------------------

if __name__ == "__main__":
    # Step 1
    game_logs_df, team_metrics_df = pull_all_game_and_team_data()

    # Step 2
    merged_games = merge_team_metrics(create_game_level_data(game_logs_df), team_metrics_df)

    # Step 3
    merged_games = add_rolling_averages(merged_games)

    # Step 4
    game_logs_df = compute_rest_days(game_logs_df)
    merged_games = merge_rest_b2b(merged_games, game_logs_df)

    # Step 5
    starters_dict = get_true_starters()
    merged_games = detect_missing_starters(merged_games, starters_dict)

    # Final Save
    merged_games.to_csv("nba_master_dataset_2015_2025.csv", index=False)
    print("\nAll NBA data pulled, processed, and saved to nba_master_dataset_2015_2025.csv")


Fetching game logs for 2015-16...
Fetching team metrics for 2015-16...
✅ Completed season 2015-16
Fetching game logs for 2016-17...
Fetching team metrics for 2016-17...
✅ Completed season 2016-17
Fetching game logs for 2017-18...
Fetching team metrics for 2017-18...
✅ Completed season 2017-18
Fetching game logs for 2018-19...
Fetching team metrics for 2018-19...
✅ Completed season 2018-19
Fetching game logs for 2019-20...
Fetching team metrics for 2019-20...
✅ Completed season 2019-20
Fetching game logs for 2020-21...
Fetching team metrics for 2020-21...
✅ Completed season 2020-21
Fetching game logs for 2021-22...
Fetching team metrics for 2021-22...
✅ Completed season 2021-22
Fetching game logs for 2022-23...
Fetching team metrics for 2022-23...
✅ Completed season 2022-23
Fetching game logs for 2023-24...
Fetching team metrics for 2023-24...
✅ Completed season 2023-24
Fetching game logs for 2024-25...
Fetching team metrics for 2024-25...
✅ Completed season 2024-25
Fetching game logs f

KeyboardInterrupt: 

In [63]:
merged_games.to_csv("nba_master_dataset_2015_2025.csv", index=False)

In [61]:
# Check if KeyPlayersOut columns exist
print(merged_games.columns)


Index(['GameID', 'Season', 'GameDate', 'HomeTeam', 'AwayTeam', 'HomePTS',
       'AwayPTS', 'TotalPoints', 'Home_TEAM_NAME_x', 'Home_TEAM_ID',
       ...
       'Away_Rolling_TOVPct_10g', 'Away_Rolling_REBPct_10g',
       'Home_TEAM_NAME_y', 'Home_GAME_DATE', 'Home_Rest_Days',
       'Home_BackToBack', 'Away_TEAM_NAME_y', 'Away_GAME_DATE',
       'Away_Rest_Days', 'Away_BackToBack'],
      dtype='object', length=114)
